# Lab 5.3 – Auffälligkeiten erkennen

**Beispiel (fertige Lösung).** Zellen der Reihe nach mit **Umschalt+Enter** ausführen.

**Regel:** Ein Tag ist auffällig, wenn der Tagesumsatz einer Region mehr als `schwelle` Standardabweichungen vom Mittelwert dieser Region im selben Monat abweicht. Ergebnis: Funktion `find_anomalies(df, schwelle=3)`, Diagramm und `auffaellige_tage.csv` in `ORDNER`.

In [1]:
import pandas as pd
import plotly.express as px

EINGABE = "project/examples/m02-data-preparation/l03-features/umsatz_features.csv"
ORDNER = "project/examples/m05-ml-basics/l03-anomalies/"     # hier landet auffaellige_tage.csv

df = pd.read_csv(EINGABE, parse_dates=["datum"])

# Tagesumsatz je Region: eine Zeile je Tag und Region
tage = df.groupby(["datum", "region"], as_index=False)["umsatz"].sum()
tage["monat"] = tage["datum"].dt.strftime("%Y-%m")
print(len(tage), "Zeilen (Tag × Region)")
tage.head()

2872 Zeilen (Tag × Region)


,datum,region,umsatz,monat
0,2024-01-01,Nord,5319.33,2024-01
1,2024-01-01,Ost,5653.99,2024-01
2,2024-01-01,Süd,2033.16,2024-01
3,2024-01-01,West,3256.15,2024-01
4,2024-01-02,Nord,2880.42,2024-01


## Schritt 1 (fertig): Mittelwert und Standardabweichung je Region und Monat

`transform` schreibt den Wert der Gruppe (Region + Monat) in jede Zeile dieser Gruppe. So steht neben jedem Tag der Ø Tagesumsatz seiner Region im selben Monat und die typische Abweichung davon.

In [2]:
gruppe = tage.groupby(["region", "monat"])["umsatz"]
tage["mittel"] = gruppe.transform("mean")      # Ø Tagesumsatz der Region in diesem Monat
tage["streuung"] = gruppe.transform("std")     # Standardabweichung: typische Abweichung vom Ø

# Beispiel: Region West, November 2025
tage[(tage["region"] == "West") & (tage["monat"] == "2025-11")].round(0).tail(4)

,datum,region,umsatz,monat,mittel,streuung
1987,2025-11-25,West,8268.0,2025-11,5517.0,4807.0
1991,2025-11-26,West,2881.0,2025-11,5517.0,4807.0
1995,2025-11-27,West,25329.0,2025-11,5517.0,4807.0
1999,2025-11-28,West,4276.0,2025-11,5517.0,4807.0


## Schritt 2: Regel als Funktion (Aufgabe 1)

Die Zelle darunter mit dem vorgegebenen Prompt aus der Übung (Aufgabe 3) füllen, dann die Prüfzelle ausführen.

In [3]:
# >>> AUFGABE 1: Funktion find_anomalies(df, schwelle=3) schreiben – Prompt siehe Übung, Aufgabe 3
def find_anomalies(df, schwelle=3):
    """Markiert Tage, die mehr als schwelle Standardabweichungen vom Monatsmittel abweichen."""
    df = df.copy()
    df["abstand"] = (df["umsatz"] - df["mittel"]) / df["streuung"]   # in Standardabweichungen
    df["auffaellig"] = df["abstand"].abs() > schwelle                   # nach oben oder unten
    return df
# <<< ENDE AUFGABE 1

In [4]:
# Prüfzelle – nicht ändern
def pruefe_anomalien():
    if "find_anomalies" not in globals():
        return "nicht OK – Funktion find_anomalies fehlt noch (Aufgabe 1)."
    ergebnis = find_anomalies(tage)
    if not {"abstand", "auffaellig"} <= set(ergebnis.columns):
        return "nicht OK – Spalten abstand und auffaellig fehlen im Ergebnis."
    treffer = ergebnis[ergebnis["auffaellig"]]
    bekannt = [("2025-11-27", "West"), ("2026-02-10", "Ost")]
    fehlt = [f"{d} {r}" for d, r in bekannt if not ((treffer["datum"] == d) & (treffer["region"] == r)).any()]
    if fehlt:
        return "nicht OK – nicht gefunden: " + ", ".join(fehlt) + ". abstand = (umsatz - mittel) / streuung."
    if len(treffer) != 11:
        return f"nicht OK – {len(treffer)} statt 11 auffällige Tage bei Schwelle 3: Betrag des Abstands nehmen (abs)."
    if find_anomalies(tage, schwelle=4)["auffaellig"].sum() != 2:
        return "nicht OK – der Parameter schwelle wird nicht verwendet (Schwelle 4 muss 2 Tage liefern)."
    return "OK – 11 auffällige Tage bei Schwelle 3, darunter 27.11.2025 West und 10.02.2026 Ost"


print(pruefe_anomalien())

OK – 11 auffällige Tage bei Schwelle 3, darunter 27.11.2025 West und 10.02.2026 Ost


## Schritt 3 (fertig): Diagramm und Liste der auffälligen Tage

In [5]:
ergebnis = find_anomalies(tage, schwelle=3)
auffaellig = ergebnis[ergebnis["auffaellig"]]

fig = px.line(ergebnis, x="datum", y="umsatz", color="region",
              title="11 auffällige Tage bei Schwelle 3 – zwei ragen weit heraus",
              labels={"datum": "Tag", "umsatz": "Tagesumsatz in €", "region": "Region"})
fig.add_scatter(x=auffaellig["datum"], y=auffaellig["umsatz"], mode="markers", name="auffällig",
                marker=dict(size=12, color="red", symbol="circle-open", line=dict(width=3)))   # rote Kreise
fig

In [6]:
liste = auffaellig[["datum", "region", "umsatz", "mittel", "streuung", "abstand"]].round(1)
liste.to_csv(ORDNER + "auffaellige_tage.csv", index=False)
print("Gespeichert:", ORDNER + "auffaellige_tage.csv")
liste.sort_values("abstand", ascending=False)

Gespeichert: project/examples/m05-ml-basics/l03-anomalies/auffaellige_tage.csv


,datum,region,umsatz,mittel,streuung,abstand
2205,2026-02-10,Ost,33969.2,4206.4,7029.4,4.2
1995,2025-11-27,West,25329.5,5517.1,4806.7,4.1
331,2024-04-24,West,6505.7,3176.3,982.3,3.4
634,2024-08-08,Süd,6966.4,3350.6,1100.0,3.3
872,2024-10-31,Nord,7773.6,4114.0,1144.5,3.2
2853,2026-09-24,Ost,6641.0,2681.6,1223.0,3.2
71,2024-01-24,West,5713.0,3264.2,781.5,3.1
175,2024-02-29,West,6366.2,3137.4,1057.0,3.1
2786,2026-09-01,Süd,7185.3,3779.0,1095.4,3.1
1105,2025-01-21,Ost,5553.3,2827.5,896.6,3.0


## Erweiterung: andere Schwellen, Fehlalarme zählen

Prompt und Lösung: Übung, Erweiterung.

In [7]:
bekannt = [("2025-11-27", "West"), ("2026-02-10", "Ost")]    # die zwei dokumentierten Fälle
for schwelle in [2, 3, 4]:
    treffer = find_anomalies(tage, schwelle)
    treffer = treffer[treffer["auffaellig"]]
    gefunden = sum(((treffer["datum"] == d) & (treffer["region"] == r)).any() for d, r in bekannt)
    print(f"Schwelle {schwelle}: {len(treffer):>3} Treffer, davon {gefunden} dokumentiert, "
          f"{len(treffer) - gefunden:>3} Fehlalarme")

Schwelle 2: 114 Treffer, davon 2 dokumentiert, 112 Fehlalarme
Schwelle 3:  11 Treffer, davon 2 dokumentiert,   9 Fehlalarme
Schwelle 4:   2 Treffer, davon 2 dokumentiert,   0 Fehlalarme
